# Prepare raw and processed data for GEO 

Xenium GEO Submission File Checklist from CompBio Ops. Zip the following in a single folder for GEO upload:

Raw files:

1. morphology.ome.tif
2. Transcripts.parquet

Processed files:

3. cell_feature_matrix (barcodes.tsv.gz, features.tsv.gz, matrix.mtx.gz OR cell_feature_matrix.h5)
4. cells.parquet
5. cell_boundaries.parquet
6. nucleus_boundaries.parquet
7. Seurat object .rds / AnnData object .h5ad processed file (optional, if it was generated)

Metadata:

8. GEO metadata sheet 


In [2]:
from pathlib import Path
import os
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import sparse
import warnings
import session_info
import sys

## Local file info

In [3]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import DATA_DIR, BASE_OUTDIR

# xenium outputs location after adding prefixes
source_dir = DATA_DIR
dest_dir = DATA_DIR / "geo_organized"
raw_dir = dest_dir / "raw"
processed_dir = dest_dir / "processed"

# processed adata location
out_dir = BASE_OUTDIR / "downstream_analysis"

# zip folder name - this is the folder to consolidate all files within 
dest_dir_sub = "bangsetal_geo_20260626"

## 1. Copy all files to single folder for geo upload
- Copy contents of dest_dir and raw_dir into dest_dir_sub
- Copy adata_HDM_processed.h5ad from out_dir to dest_dir_sub


In [9]:
import shutil

# Single folder that will hold all files for the GEO upload
geo_folder = dest_dir / dest_dir_sub
geo_folder.mkdir(parents=True, exist_ok=True)

# MOVE the organized raw and processed files into the single upload folder
for src in [raw_dir, processed_dir]:
    for f in sorted(src.iterdir()):
        target = geo_folder / f.name
        
        # This replaces the entire if f.is_dir() / copytree / copy2 block:
        shutil.move(str(f), str(target))




# COPY the processed AnnData object
shutil.copy2(out_dir / "adata_HDM_processed.h5ad", geo_folder / "adata_HDM_processed.h5ad")

# Verify contents
print(f"Files in {geo_folder}:")
for f in sorted(geo_folder.iterdir()):
    print("  ", f.name)

Files in /home/workspace/data/temp/mouse_lung/geo_organized/bangsetal_geo_20260626:
   TIS08778_cell_boundaries.parquet
   TIS08778_cell_feature_matrix.h5
   TIS08778_cells.parquet
   TIS08778_morphology.ome.tif
   TIS08778_nucleus_boundaries.parquet
   TIS08778_transcripts.parquet
   TIS08779_cell_boundaries.parquet
   TIS08779_cell_feature_matrix.h5
   TIS08779_cells.parquet
   TIS08779_morphology.ome.tif
   TIS08779_nucleus_boundaries.parquet
   TIS08779_transcripts.parquet
   TIS08780_cell_boundaries.parquet
   TIS08780_cell_feature_matrix.h5
   TIS08780_cells.parquet
   TIS08780_morphology.ome.tif
   TIS08780_nucleus_boundaries.parquet
   TIS08780_transcripts.parquet
   TIS08781_cell_boundaries.parquet
   TIS08781_cell_feature_matrix.h5
   TIS08781_cells.parquet
   TIS08781_morphology.ome.tif
   TIS08781_nucleus_boundaries.parquet
   TIS08781_transcripts.parquet
   adata_HDM_processed.h5ad


## 2. Zip
- Then zip using shutil, zip the folder

In [4]:
import shutil

# Zip the consolidated folder -> <dest_dir>/<dest_dir_sub>.zip
zip_path = shutil.make_archive(
    base_name=str(dest_dir / dest_dir_sub),  # output path without the .zip extension
    format="zip",
    root_dir=str(dest_dir),                  # archive paths are relative to here
    base_dir=dest_dir_sub,                   # the folder to zip
)
print("Created zip:", zip_path)

Created zip: /home/workspace/data/temp/mouse_lung/geo_organized/bangsetal_geo_20260626.zip


## 3. Gsutil upload to HISE

#### In IDE terminal:
gcloud auth login
#### Follow the prompts to authenticate your login in the browser and then proceed. You will have to follow the prompts in the browser and copy the required password into the terminal

#### After login is complete:
gcloud components install gsutil
#### Then follow prompts as applicable

#### Command to copy to project store:
nohup gsutil -m cp /home/workspace/data/temp/mouse_lung/geo_organized/bangsetal_geo_20260626.zip gs://wftini_mouseaifi-219b3400-2a33-483a-a457-d5272e41d7db
#### Once upload is complete, the nohup.out file will stop updating. Time taken for upload will depend on size of zip file

# 4. HISE uuid after completion


c2d5c294-a93b-42e2-83e5-e2b96ccf7c49

# Re-upload most updated processed adata object (July 23, 2026)

In [4]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import DATA_DIR, BASE_OUTDIR

# xenium outputs location after adding prefixes
source_dir = DATA_DIR
dest_dir = DATA_DIR / "geo_organized"
raw_dir = dest_dir / "raw"
processed_dir = dest_dir / "processed"

# processed adata location
out_dir = BASE_OUTDIR / "downstream_analysis"

# zip folder name - this is the folder to consolidate all files within 
dest_dir_sub_update = "bangsetal_geo_20260626_update"

In [6]:
geo_folder_update = dest_dir / dest_dir_sub_update
print('making dir ', geo_folder_update)
geo_folder_update.mkdir(parents=True, exist_ok=True)


making dir  /home/workspace/data/temp/mouse_lung/geo_organized/bangsetal_geo_20260626_update


In [8]:
import shutil
shutil.copy2(out_dir / "adata_HDM_processed.h5ad", geo_folder_update / "adata_HDM_processed.h5ad")

PosixPath('/home/workspace/data/temp/mouse_lung/geo_organized/bangsetal_geo_20260626_update/adata_HDM_processed.h5ad')

nohup gsutil -m cp /home/workspace/data/temp/mouse_lung/geo_organized/bangsetal_geo_20260626_update/adata_HDM_processed.h5ad gs://wftini_mouseaifi-219b3400-2a33-483a-a457-d5272e41d7db

HISE uuid: c21e8548-860c-4dca-8b12-336007183382